In [1]:
import pandas as pd
from sqlalchemy import create_engine


# 1. DB 연결
user = "root"
password = ""
host = "localhost"
port = 3306
database = "vigloo_youtube_db"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4"
)


# 2. CSV 불러오기
vigloo = pd.read_csv("vigloo_all_works.csv")
youtube = pd.read_csv("youtube_video_comment_final.csv")

# 3. 불필요 컬럼 제거
vigloo = vigloo.drop(columns=["순번"], errors="ignore")

# 4. 컬럼명 변경
vigloo = vigloo.rename(columns={
    "제목": "title",
    "태그/설명": "tag_desc",
    "국가": "country",
    "장르": "genre",
    "줄거리": "story",
    "총회차": "total_episodes",
    "URL": "url"
})

youtube = youtube.rename(columns={
    "작품명": "title",
    "검색키워드": "search_keyword",
    "영상제목": "video_title",
    "채널명": "channel_name",
    "업로드일": "upload_date",
    "조회수": "views",
    "좋아요수": "likes",
    "댓글수": "comment_count",
    "영상URL": "video_url",
    "댓글작성자": "comment_author",
    "댓글내용": "comment_text",
    "댓글좋아요수": "comment_likes",
    "댓글작성일": "comment_date"
})

# 5. 데이터 정리

# 문자열 정리 
vigloo["title"] = vigloo["title"].astype(str).str.strip()
youtube["title"] = youtube["title"].astype(str).str.strip()

# 숫자형
vigloo["total_episodes"] = pd.to_numeric(
    vigloo["total_episodes"], errors="coerce"
).fillna(0).astype(int)

for col in ["views", "likes", "comment_count", "comment_likes"]:
    youtube[col] = pd.to_numeric(youtube[col], errors="coerce")

# 날짜형
youtube["upload_date"] = pd.to_datetime(youtube["upload_date"], errors="coerce")
youtube["comment_date"] = pd.to_datetime(youtube["comment_date"], errors="coerce")

# 6. vigloo 테이블 저장
vigloo.to_sql("vigloo", con=engine, if_exists="append", index=False)

# 7. work_id 가져오기
works = pd.read_sql("SELECT work_id, title FROM vigloo", con=engine)

# 8. title 기준 매핑
youtube = youtube.merge(works, on="title", how="left")

# 9. 매칭 안된 데이터 확인
missing = youtube[youtube["work_id"].isna()]["title"].unique()

if len(missing) > 0:
    print("매칭 안된 작품:")
    print(missing)
else:
    print("모든 데이터 매칭 완료")

# 10. work_id 정리
youtube["work_id"] = youtube["work_id"].astype(int)

# 11. youtube 테이블 저장
youtube.to_sql("youtube", con=engine, if_exists="append", index=False)

print("SQL 업로드 완료")

모든 데이터 매칭 완료
SQL 업로드 완료
